# GPT-2 Inference Benchmark
Measures tokens/sec, time to first token, and peak GPU memory for GPT-2 small on a T4,
comparing fp32 vs fp16 and KV cache on vs off at batch sizes 1, 8, and 32.

In [ ]:
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
tok.padding_side = "left"

In [2]:
def bench(model, batch_size, use_cache=True, prompt="The future of physics is", n_tokens=300, runs=10, warmup=10):
    inputs = tok([prompt] * batch_size, return_tensors="pt", padding=True).to("cuda")
    kw = dict(do_sample=False, pad_token_id=tok.eos_token_id, use_cache=use_cache)
    full = dict(max_new_tokens=n_tokens, min_new_tokens=n_tokens, **kw)

    for i in range(warmup):  #warm-up
        model.generate(**inputs, **full)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    t = time.time()  #ttft
    for i in range(runs):
        model.generate(**inputs, max_new_tokens=1, **kw)
    torch.cuda.synchronize()
    ttft_ms = (time.time() - t) / runs * 1000

    t = time.time()  #tokens/second
    for i in range(runs):
        model.generate(**inputs, **full)
    torch.cuda.synchronize()
    tps = batch_size * n_tokens * runs / (time.time() - t)

    return ttft_ms, tps, torch.cuda.max_memory_allocated() / 1e6  #VRAM

In [3]:
configs = [
    ("fp32, cache on",  torch.float32, True),
    ("fp32, cache off", torch.float32, False),
    ("fp16, cache on",  torch.float16, True),
    ("fp16, cache off", torch.float16, False),
]

print("| config | batch | TTFT (ms) | tokens/sec | peak mem (MB) |")
print("|---|---|---|---|---|")
for name, dtype, cache in configs:
    model = AutoModelForCausalLM.from_pretrained("gpt2", dtype=dtype).to("cuda")
    for bs in [1, 8, 32]:
        ttft, tps, mem = bench(model, bs, use_cache=cache)
        print(f"| {name} | {bs} | {ttft:.1f} | {tps:.0f} | {mem:.0f} |")
    del model
    torch.cuda.empty_cache()

| config | batch | TTFT (ms) | tokens/sec | peak mem (MB) |
|---|---|---|---|---|


model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

| fp32, cache off | 1 | 10.4 | 73 | 530 |
| fp32, cache off | 8 | 11.4 | 94 | 677 |
| fp32, cache off | 32 | 17.1 | 99 | 1179 |


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

| fp16, cache on | 1 | 11.2 | 103 | 280 |
| fp16, cache on | 8 | 12.8 | 811 | 370 |
| fp16, cache on | 32 | 12.6 | 3121 | 662 |


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

| fp16, cache off | 1 | 11.0 | 108 | 278 |
| fp16, cache off | 8 | 11.5 | 359 | 354 |
| fp16, cache off | 32 | 10.8 | 388 | 614 |


## Perplexity
Checks that fp16 does not hurt quality. WikiText-2 test set non-overlapping 1024-token chunks (lower is better).

In [4]:
import math
from datasets import load_dataset

test = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
enc = tok("\n\n".join(test["text"]), return_tensors="pt").input_ids

@torch.no_grad()
def perplexity(model, max_len=1024):
    nll, count = 0.0, 0
    for i in range(0, enc.size(1) - max_len, max_len):
        ids = enc[:, i:i + max_len].to("cuda")
        loss = model(ids, labels=ids).loss
        nll += loss.float().item() * (max_len - 1)
        count += max_len - 1
    return math.exp(nll / count)

for name, dtype in [("fp32", torch.float32), ("fp16", torch.float16)]:
    m = AutoModelForCausalLM.from_pretrained("gpt2", dtype=dtype).to("cuda")
    print(name, "perplexity:", round(perplexity(m), 2))
    del m
    torch.cuda.empty_cache()

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


fp32 perplexity: 29.95


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

fp16 perplexity: 29.96


## torch.compile
Uses a static KV cache (which makes Hugging Face compile the forward pass automatically) to cut per-step CPU overhead. I skipped the CUDA graphs (the in-place cache update conflicts with them), so the speedup comes from kernel fusion.

In [5]:
model = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float16).to("cuda")
model.generation_config.cache_implementation = "static"

for bs in [1, 8, 32]:
    ttft, tps, mem = bench(model, bs)
    print(f"| fp16, static cache, auto-compiled | {bs} | {ttft:.1f} | {tps:.0f} | {mem:.0f} |")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

W1027 19:39:41.869000 640 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


| fp16, static cache, auto-compiled | 1 | 13.4 | 353 | 280 |
| fp16, static cache, auto-compiled | 8 | 14.8 | 2230 | 369 |
| fp16, static cache, auto-compiled | 32 | 14.9 | 5338 | 659 |


## Correctness Check

In [6]:
ref = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float16).to("cuda")
inp = tok("The future of physics is", return_tensors="pt").to("cuda")
kw = dict(max_new_tokens=50, do_sample=False, pad_token_id=tok.eos_token_id)

a = ref.generate(**inp, **kw)
b = model.generate(**inp, **kw)
print("identical output:", torch.equal(a, b))

import gc
del ref, model
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

identical output: True


## Long prompt (prefill)

In [7]:
long_prompt = "The future of physics is " * 100   # approx 500 tokens

print("| config | prompt | TTFT (ms) | tokens/sec | peak mem (MB) |")
print("|---|---|---|---|---|")
for name, dtype in [("fp32", torch.float32), ("fp16", torch.float16)]:
    m = AutoModelForCausalLM.from_pretrained("gpt2", dtype=dtype).to("cuda")
    for bs in [1, 8]:
        ttft, tps, mem = bench(m, bs, prompt=long_prompt, n_tokens=100)
        print(f"| {name}, batch {bs} | ~500 tok | {ttft:.1f} | {tps:.0f} | {mem:.0f} |")
    del m
    torch.cuda.empty_cache()

| config | prompt | TTFT (ms) | tokens/sec | peak mem (MB) |
|---|---|---|---|---|


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

| fp32, batch 1 | ~500 tok | 34.1 | 101 | 588 |
| fp32, batch 8 | ~500 tok | 289.8 | 480 | 1077 |


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

| fp16, batch 1 | ~500 tok | 13.4 | 99 | 301 |
| fp16, batch 8 | ~500 tok | 70.7 | 746 | 554 |


## Profiling

In [8]:
import time
from torch.profiler import profile, ProfilerActivity

m = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float16).to("cuda")
inputs = tok(["The future of physics is"], return_tensors="pt", padding=True).to("cuda")
kw = dict(max_new_tokens=50, min_new_tokens=50, do_sample=False, pad_token_id=tok.eos_token_id)

for _ in range(5):          # warmup
    m.generate(**inputs, **kw)
torch.cuda.synchronize()

t = time.time()
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    m.generate(**inputs, **kw)
    torch.cuda.synchronize()
wall_ms = (time.time() - t) * 1000

print(f"wall time for 50 tokens: {wall_ms:.0f} ms")
print(prof.key_averages().table(sort_by="self_device_time_total", row_limit=10))

del m
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


wall time for 50 tokens: 1318 ms
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                            aten::addmm        18.86%      96.487ms        25.38%     129.856ms      54.107us      55.801ms        43.83%      55.801ms      23.250us          2400  
                                               aten::mm         0.38%       1.939ms         0.54%       2.754ms      55.077us      19.072ms        14.98%      19.072ms     38

## INT8 quantization

In [9]:
!pip install -q bitsandbytes

import logging
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

from transformers import BitsAndBytesConfig

cfg = BitsAndBytesConfig(load_in_8bit=True)
m8 = AutoModelForCausalLM.from_pretrained(
    "gpt2", quantization_config=cfg, device_map="cuda", dtype=torch.float16
)

print("| config | batch | TTFT (ms) | tokens/sec | peak mem (MB) |")
print("|---|---|---|---|---|")
for bs in [1, 8]:
    ttft, tps, mem = bench(m8, bs, n_tokens=100, runs=3, warmup=1)
    print(f"| int8 (bitsandbytes), cache on | {bs} | {ttft:.1f} | {tps:.0f} | {mem:.0f} |")

print("int8 perplexity:", round(perplexity(m8), 2))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

| config | batch | TTFT (ms) | tokens/sec | peak mem (MB) |
|---|---|---|---|---|
| int8 (bitsandbytes), cache on | 1 | 46.2 | 21 | 181 |
| int8 (bitsandbytes), cache on | 8 | 43.7 | 222 | 220 |
int8 perplexity: 30.11
